In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from TTS.utils.manage import ModelManager
from trainer import Trainer, TrainerArgs
from TTS.tts.configs.vits_config import VitsConfig, VitsAudioConfig
from TTS.tts.configs.shared_configs import CharactersConfig
from TTS.tts.models.vits import Vits
from TTS.utils.audio import AudioProcessor
from trainer.callbacks import TrainerCallback
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

# Configuration
class Config:
    data_path = "./dataset/chunks"
    output_path = "./tts_output"
    pretrained_model = "tts_models/en/ljspeech/vits"
    batch_size = 6
    eval_batch_size = 2
    num_loader_workers = 2
    max_audio_len = 15
    lr = 1e-5
    epochs = 100
    steps_to_generate = 3600
    save_step = 1800
    log_step = 100
    save_n_checkpoints = 5

def preprocess_data(config):
    """Load and format dataset with integrity checks"""
    metadata = []
    for file in os.listdir(config.data_path):
        if file.endswith(".csv"):
            csv_path = os.path.join(config.data_path, file)
            try:
                df = pd.read_csv(csv_path, names=["file", "text"])
                for _, row in df.iterrows():
                    wav_path = os.path.join(config.data_path, row["file"])
                    if os.path.exists(wav_path) and wav_path.endswith(".wav"):
                        metadata.append({"text": row["text"], "audio_file": wav_path})
            except Exception as e:
                print(f"Error processing {csv_path}: {str(e)}")
    print(f"Loaded {len(metadata)} valid samples")
    return metadata

class SampleGenerationCallback(TrainerCallback):
    """Callback to generate audio samples during training"""
    def __init__(self, config, ap):
        self.config = config
        self.ap = ap
        self.test_phrases = [
            "This is a test synthesis",
            "The quick brown fox jumps over the lazy dog",
            "Fine tuning in progress for audiobook narration"
        ]
    
    def on_train_step_end(self, trainer, model, outputs):
        """Generate samples at specified intervals"""
        if trainer.global_step % self.config.steps_to_generate == 0:
            print(f"\nGenerating samples at step {trainer.global_step}")
            model.eval()
            for i, text in enumerate(self.test_phrases):
                # Use model's inference directly
                output = model.inference(text)
                wav = output["wav"].squeeze().cpu().numpy()
                
                out_path = os.path.join(
                    trainer.output_path,
                    f"step_{trainer.global_step}_sample_{i}.wav"
                )
                self.ap.save_wav(wav, out_path, self.ap.sample_rate)
                print(f"Saved sample to {out_path}")
            model.train()


def main():
    config = Config()
    os.makedirs(config.output_path, exist_ok=True)
    # Audio configuration
    audio_config = VitsAudioConfig(
        sample_rate=22050,
        num_mels=80,
        fft_size=2048,
        win_length=1100,
        hop_length=256,
        mel_fmin=0,
        mel_fmax=8000,
    )
    
    # Audio processor setup
    ap = AudioProcessor(**audio_config)
    
    # Prepare dataset
    metadata = preprocess_data(config)
    if len(metadata) == 0:
        raise ValueError("No valid audio/text pairs found!")
    train_samples, eval_samples = train_test_split(
        metadata, test_size=0.1, random_state=42
    )
    print(f"Training samples: {len(train_samples)}, Eval samples: {len(eval_samples)}")
    
    # Model configuration
    characters_config = CharactersConfig(
        characters="ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz!'(),-.:;? ",
        vocab_dict=None,
    )
    model_config = VitsConfig(
        batch_size=config.batch_size,
        eval_batch_size=config.eval_batch_size,
        num_loader_workers=config.num_loader_workers,
        run_eval=True,
        epochs=config.epochs,
        lr=config.lr,
        audio=audio_config,
        use_speaker_embedding=False,
        text_cleaner="english_cleaners",
        max_audio_len=config.max_audio_len,
        # LJSpeech tokenizer parameters
        characters=characters_config,
        add_blank=True,
    )
    model_config.save_json(os.path.join(config.output_path, "config.json"))

    # Initialize model with direct state dict loading
    manager = ModelManager()
    model_path, config_path, _ = manager.download_model(config.pretrained_model)
    model = Vits.init_from_config(model_config)
    
    # Load state dictionary directly
    state_dict = torch.load(model_path, map_location="cpu")
    if "model" in state_dict:
        state_dict = state_dict["model"]
    model.load_state_dict(state_dict, strict=False)
    
    # Initialize trainer
    trainer_args = TrainerArgs(
        output_path=config.output_path,
        save_step=config.save_step,
        log_step=config.log_step,
        epochs=config.epochs,
        mixed_precision=False,
        use_cuda=torch.backends.mps.is_available(),
        save_n_checkpoints=config.save_n_checkpoints,
        save_best=True,
        print_step=50,
        print_eval=False,
    )
    
    trainer = Trainer(
        trainer_args,
        config=model_config,
        output_path=config.output_path,
        model=model,
        train_samples=train_samples,
        eval_samples=eval_samples,
        callbacks=[SampleGenerationCallback(config, ap)],
    )
    
    # Start training
    print(f"Starting training with {len(train_samples)} samples...")
    trainer.fit()

if __name__ == "__main__":
    main()

 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db:0
 | > frame_shift_ms:None
 | > frame_length_ms:None
 | > ref_level_db:None
 | > fft_size:2048
 | > power:None
 | > preemphasis:0.0
 | > griffin_lim_iters:None
 | > signal_norm:None
 | > symmetric_norm:None
 | > mel_fmin:0
 | > mel_fmax:8000
 | > pitch_fmin:None
 | > pitch_fmax:None
 | > spec_gain:20.0
 | > stft_pad_mode:reflect
 | > max_norm:1.0
 | > clip_norm:True
 | > do_trim_silence:False
 | > trim_db:60
 | > do_sound_norm:False
 | > do_amp_to_db_linear:True
 | > do_amp_to_db_mel:True
 | > do_rms_norm:False
 | > db_level:None
 | > stats_path:None
 | > base:10
 | > hop_length:256
 | > win_length:1100
Loaded 1872 valid samples
Training samples: 1684, Eval samples: 188
 > tts_models/en/ljspeech/vits is already downloaded.
 > Setting up Audio Processor...
 | > sample_rate:22050
 | > resample:False
 | > num_mels:80
 | > log_func:np.log10
 | > min_level_db

TypeError: 'NoneType' object is not iterable